# पाठ १८ (पछिल्लो): रसिदहरू जसले प्रमाणित गर्छ कि *मान्छे* ले कार्यलाई अनुमति दिए

यस पाठले प्रमाणित गर्छ कि **एजेन्ट** ले के गर्यो र **गेट** ले के निर्णय गर्यो। यो नोटबुकले हराएको अर्धांश थप्दछ: प्रमाण कि एक **नामित मान्छेले** **ठ्याक्कै** गरिएको कार्यलाई अनुमति दियो — सम्पूर्ण क्यानोनिकल कार्यमा मान्छे-द्वारा राखिएको छुट्टै हस्ताक्षर, जुन अफलाइन जाँच गरिएको छ।

यहाँ दुई वटा सामग्रीहरूले **पाठका रसिदहरू जस्तै नै लिफाफा संरचना** प्रयोग गर्छन्: एउटा सिधा पेलोड जसमा `type` फिल्ड हुन्छ, जुन Ed25519 द्वारा क्यानोनिकल JCS बाइटहरूमा सिधै हस्ताक्षर गरिएको हुन्छ, र संगठित `signature` वस्तु जोडिएको (र हस्ताक्षर गरिएका बाइटहरूबाट अलग गरिएको)। अनुमोदन रसिद नयाँ `type` (`human.approval.v1`) हो जुन कार्य प्रकारसँगै हुन्छ, जसले एकै `verify_chain` ले दुबै प्रकारका सामग्रीलाई मुख्य नोटबुकमा बनाइएको समान कोड पथ प्रयोग गरेर समेट्छ। यो मानव-अनुमोदन रसिद यहाँ परिभाषित शैक्षिक संरचना हो, जुन draft-farley-acta-signed-receipts द्वारा परिभाषित रसिद प्रकार होइन।

मुख्य नोटबुकमा देखाइएको डेमो भेरिफायरभन्दा एउटा जानाजानी गरिएको उन्नति: भेरिफायरले यहाँ `signature.key_id` लाई रसिदभित्र रहेको सार्वजनिक कुञ्जीमा विश्वास नगरी **पिन गरिएको कुञ्जी रजिष्ट्रि**सँग सुल्झाउँछ। यो नै उत्पादन स्थिती हो जुन पाठको आफ्नै चेकलिष्टले सिफारिस गर्छ ("प्रमाणीकरण सार्वजनिक कुञ्जी प्रकाशित गर्नुहोस्"), र यसैले नक्कली बनाउन नकार हो, आफ्नै कुञ्जी ल्याउने भाँडोलाई रोक्छ।

यो नोटबुकले सिकाउने नियम: **हस्ताक्षर गरिएको अनुमोदन आफैंमा अधिकार होइन।** अधिकार केवल जब अनुमोदन रसिद र कार्य रसिद एकै क्यानोनिकल कार्यसँग आबद्ध हुन्छन् कार्यान्वयन समयमा, एउटा नीति संस्करण, कुञ्जी, र म्याद जुन अझै वैध छ, र अनुमोदन जसलाई अझै प्रयोग गरिएको छैन। प्रत्येक असफलताले **भिन्न कारण** सहित अस्वीकृत गर्छ, जसले तपाईंलाई *अधिकार म्याद सकियो* र *कार्यान्वयन गरिएको कार्य परिवर्तन भयो* छुट्याउन मद्दत गर्दछ।


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## ठ्याक्कै कार्य

स्वीकृतिको इकाई **क्यानोनिकल कार्य वस्तु** हो — "रिफन्ड स्वीकृत गर्नुहोस्" जस्तो अस्पष्ट लेबल होइन, तर ठ्याक्कै, पूर्ण रूपमा निर्दिष्ट गरिएको कार्य हो। सम्पूर्ण वस्तुमा हस्ताक्षर गर्नु (र त्यसबाट एक डाइजेस्ट निकाल्नु) नै पछि हामीले प्रमाणित गर्न सक्छौं कि मान्छेले *यो* मात्र स्वीकृत गर्‍यो र अरू केही होइन।


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## एउटा लिफाफा, दुई अधिकारहरू

हरेक रसिद पाठको लिफाफा हो: एउटा फ्ल्याट पेलोड जुन `type` फिल्ड हुन्छ, साथै एक `signature` वस्तु (`alg`, `sig`, `key_id`) हुन्छ जुन **साईन्ड गरिएको बाइट्सको भाग होइन**। `verify_envelope` भनेको दुवै प्रकारका रसिदका लागि साझा संरचनागत + हस्ताक्षर जाँच हो; जुन **पिन गरिएको कुञ्जी रजिष्ट्रि** यसले `signature.key_id` समाधान गर्छ, त्यो हो जुन अधिकारहरूलाई अलग राख्छ:

- **अनुमोदन रसिद** (`human.approval.v1`) — नामित अनुमोदक, पूर्ण क्यानोनिकल क्रिया **र यसको डाइजेस्ट**, `policy_version`, जारी र समाप्ति समय। एक पटकको उपभोग चेन स्तरमा ट्रयाक गरिन्छ।
- **क्रिया रसिद** (`agent.action.v1`) — एजेन्ट पहिचान, `run_id`, उही क्यानोनिकल क्रिया **डाइजेस्ट**, क्रियान्वयन नतिजा र समय, र `parent_approval_ref`: अनुमोदनको `receipt_hash`, जसरी `previous_receipt_hash` पाठको चेनमा हुन्छ।

साझा `action_digest` फिल्ड हो जुन बाँधनमा निर्भर हुन्छ। `key_id` हस्ताक्षर वस्तुमा मात्र lookup संकेतको रूपमा हुन्छ: यसलाई फरक पिन गरिएको कुञ्जीसँग फेरि इङ्गित गर्दा हस्ताक्षर जाँच असफल हुन्छ, त्यसैले यसले केही दिंदैन।


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: जहाँ बाँध्ने निर्णय वास्तवमै गरिन्छ

`verify_chain` दुई हस्ताक्षर जाँचहरू माथिको सजिलो आवरण होइन । यो एक मात्र स्थान हो जहाँ साझा क्यानोनिकल `action_digest`, नीति/सिक्का/समाप्ति **ताजगी** अनुमोदनको, र अनुमोदनको **एकपटक مصرف** सँगै जाँचिन्छ, अहिले *सञ्चालन भइरहेको* क्रियावलीको विरुद्धमा ।

प्रत्येक असफलता **भिन्न कारण**का साथ अस्वीकार गर्छ, त्यसैले अस्वीकृतिको पाठकले भन्न सक्छ कि अधिकार म्याद सकियो (नीति सारियो, सिक्रेट परिवर्तन भयो, अनुमोदन समाप्त भयो, अनुमोदन प्रयोग भयो) वा संचालन गरिएको क्रियालाई अझै मान्य अनुमोदनबाट अगाडि परिवर्तन गरिएको छ (डाइजेस्ट प्रतिस्थापन) ।


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## बाइंडिङले के समात्छ

तलका प्रत्येक केसहरू **विन्द गरिएको** हुन्छन् र **विभिन्न कारण** हुन्छन्। पहिलो ब्लक क्लासिक सेट हो (ट्याम्पर, भ्रमित डेप्युटी, रिप्ले, कुनै पनि अधिकारमा नक्कली, विकृत इनपुट)। दोस्रो ब्लक वह जो विशेषता वास्तविक बनाउँछ न कि दावी गरिएको:

- **पुरानो अधिकार** — हस्ताक्षर अझै मान्य छ, तर नीति संस्करण सरेको छ, अनुमोदक कुञ्जी पिन गरिएको रजिष्ट्रीबाट हटाइएको छ, वा स्वीकृति कार्यान्वयन अघि समाप्त भइसकेको छ;
- **डाइजेस्ट प्रतिस्थापन** — मान्य रूपमा हस्ताक्षर गरिएको कार्य रिसीप्ट जसको `parent_approval_ref` ले *वास्तविक* स्वीकृति तिर संकेत गर्दछ, तर त्यो स्वीकृतिको क्यानोनिकल कार्य डाइजेस्टले वास्तवमा कार्यान्वयन भइरहेको कार्यसँग मेल खाँदैन।


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## के प्रमाणित गर्छ — र के गर्दैन

**प्रमाणित:** एउटा नामावलीमा मानवीयले *यो सही आधिकारिक कार्य* (पूर्ण कार्य + डाइजेस्ट, पिन गरिएको रजिष्ट्रीबाट निर्णय भएको कुञ्जीले हस्ताक्षर गरिएको) स्वीकृति दिएको छ, र एजेन्टले *ठ्याक्कै सोही स्वीकृत कार्य* (समान डाइजेस्ट, `receipt_hash` द्वारा स्वीकृतिसँग बाँधिएको रसिद, पाठको आफ्नै चेन अवधारणा) कार्यान्वयन गरे — जबकि स्वीकृतिको नीति संस्करण, कुञ्जी, र समाप्ति हालका थिए, ठीक एक पटक। यदि कुनै पनि पक्ष परिवर्तन गर्छ भने, चेन बन्द हुन्छ, र अस्वीकृति कारणले तपाईंलाई **कुन** गुण फेरेको जानकारी दिन्छ: समयसिद्ध अधिकार बनाम परिवर्तन गरिएको कार्य।

**प्रमाणित गर्दैन:** कि स्वीकृति UI ले मानवीयलाई उनीहरूले हस्ताक्षर गर्दै गरेको कुरा देखाएको थियो (WYSIWYS आफैंमा समस्या हो), कि कुञ्जी कलाई घुमाउने अघि दबाब वा चोरी गरिएको थिएन, वा तलका असरहरू कार्यसँग मेल खाए। हस्ताक्षरित ≠ अधिकृत: अवैध नीतिको मान्य हस्ताक्षर, घुमाइएको कुञ्जी, म्याद सकिएको विन्डो, वा फरक डाइजेस्टले यहाँ केही प्रमाणित गर्दैन।

दुई रसिद प्रकारहरूले पाठको लिफाफा र एउटै `verify_chain` कोड पथ साझा गर्छन् उद्देश्यले: मुख्य नोटबुकमा कार्य रसिदहरूका लागि निर्मित बाँधने कोड नै मानिसको स्वीकृति जाँच गर्ने कोड हो। एउटै प्रमाणकर्ता अनुबंध, छुट्टाछुट्टै पिन गरिएको अधिकारहरू, आधिकारिक कार्य डाइजेस्टले जोडिएका छन् र अरू केही छैन।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
